# Adding the baseline library for the function to work

In [5]:
import sys
from pathlib import Path
import random
import pandas as pd

# 1. Cấu hình đường dẫn chuẩn cho ROOT và scripts/
cwd = Path.cwd()
SCRIPTS_DIR = cwd if cwd.name == "scripts" else cwd / "scripts"
ROOT_DIR = SCRIPTS_DIR.parent

for path in (str(ROOT_DIR), str(SCRIPTS_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# 2. Import các module cốt lõi
from curves.ecc import ECC, Point, FindOrder
from curves.exhaustive_search import exhaustive_search
from curves.bsgs import BSGS_ECC
from curves.pollard_rho import Pollard_Rho_ECC
from curves.pohlig_hellman import pohlig_hellman_ecc
from measure_harness import measure
from fastecdsa.curve import P256

# 3. Sanity Check: Kiểm tra 4 thuật toán ECDLP trên đường cong nhỏ y^2 = x^3 + 2x + 3 (mod 97)
test_curve = ECC(2, 3, 97)
G_test = Point(test_curve, 3, 6)
n_test = FindOrder(G_test)
k_secret = 42 % n_test  # Phải nhỏ hơn bậc của điểm G (n_test = 5 -> k_secret = 2)
A_test = k_secret * G_test

assert exhaustive_search(A_test, G_test, n_test) == k_secret, "Lỗi ở Exhaustive Search"
assert BSGS_ECC(A_test, G_test) == k_secret, "Lỗi ở BSGS"
assert Pollard_Rho_ECC(A_test, G_test) == k_secret, "Lỗi ở Pollard's Rho"
assert pohlig_hellman_ecc(A_test, G_test) == k_secret, "Lỗi ở Pohlig-Hellman"

# 4. Sanity Check: Kiểm tra OpenSSL backend & measure_harness trên NIST P-256
p256_curve = ECC(P256.a, P256.b, P256.p)
assert p256_curve.uses_openssl is True, "OpenSSL backend chưa được kích hoạt cho P-256"

G_p256 = Point(p256_curve, P256.G.x, P256.G.y)
bench_test = measure(lambda: 123456789 * G_p256, repeat=3, memory=True)
assert bench_test["wall_time_ns"] > 0 and bench_test["memory_peak_kib"] is not None

print("All imports & sanity checks PASSED!")
print(f"  - Small curve order : {n_test} | Recovered k = {k_secret}")
print(f"  - NIST P-256 OpenSSL: Active ({bench_test['wall_time_ns'] / 1e3:.2f} µs/op)")

All imports & sanity checks PASSED!
  - Small curve order : 5 | Recovered k = 2
  - NIST P-256 OpenSSL: Active (100.05 µs/op)
